In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

In [2]:
# --- Step 1: Download the two Orphadata files ---
# product1: disease cross-references (MONDO -> OrphaCode)
# product9_prev: epidemiology (incidence/prevalence per OrphaCode)
prod1_url = "https://www.orphadata.com/data/xml/en_product1.xml"
prod9_url = "https://www.orphadata.com/data/xml/en_product9_prev.xml"

r1 = requests.get(prod1_url)
r1.raise_for_status()
with open('en_product1.xml', 'wb') as f:
    f.write(r1.content)

r9 = requests.get(prod9_url)
r9.raise_for_status()
with open('en_product9_prev.xml', 'wb') as f:
    f.write(r9.content)



In [3]:
# --- Step 2: Parse product1 to build MONDO -> OrphaCode mapping ---
tree1 = ET.parse('en_product1.xml')
root1 = tree1.getroot()

mondo_to_orpha = {}
for disorder in root1.iter('Disorder'):
    orpha_code_el = disorder.find('OrphaCode')
    if orpha_code_el is None:
        continue
    orpha_code = orpha_code_el.text
    for ext_ref in disorder.iter('ExternalReference'):
        source = ext_ref.find('Source')
        reference = ext_ref.find('Reference')
        if source is not None and reference is not None and source.text == 'MONDO':
            mondo_to_orpha[reference.text] = orpha_code

print(f"Mapped {len(mondo_to_orpha)} MONDO IDs to OrphaCodes")



Mapped 9973 MONDO IDs to OrphaCodes


In [4]:
# --- Step 3: Parse product9_prev to pull incidence/prevalence per OrphaCode ---
tree9 = ET.parse('en_product9_prev.xml')
root9 = tree9.getroot()

epi_records = []
for disorder in root9.iter('Disorder'):
    orpha_code = disorder.find('OrphaCode').text
    disease_name = disorder.find('Name').text
    for prev in disorder.iter('Prevalence'):
        prev_type_el = prev.find('PrevalenceType/Name')
        prev_class_el = prev.find('PrevalenceClass/Name')
        val_moy_el = prev.find('ValMoy')
        geo_el = prev.find('PrevalenceGeographic/Name')
        status_el = prev.find('PrevalenceValidationStatus/Name')

        epi_records.append({
            'orpha_code': orpha_code,
            'disease_name': disease_name,
            'prevalence_type': prev_type_el.text if prev_type_el is not None else None,
            'prevalence_class': prev_class_el.text if prev_class_el is not None else None,
            'val_moy': float(val_moy_el.text) if val_moy_el is not None and val_moy_el.text else None,
            'geographic_area': geo_el.text if geo_el is not None else None,
            'validation_status': status_el.text if status_el is not None else None,
        })

epi_df = pd.DataFrame(epi_records)



In [5]:
# --- Step 4: Filter to annual incidence only ---
incidence_df = epi_df[epi_df['prevalence_type'] == 'Annual incidence'].copy()

# Prefer worldwide, validated estimates when a disease has multiple incidence rows
incidence_df['is_worldwide'] = incidence_df['geographic_area'] == 'Worldwide'
incidence_df['is_validated'] = incidence_df['validation_status'] == 'Validated'
incidence_df = incidence_df.sort_values(
    ['orpha_code', 'is_worldwide', 'is_validated'], ascending=[True, False, False]
)
incidence_best = incidence_df.drop_duplicates(subset='orpha_code', keep='first')



In [6]:
incidence_best

,orpha_code,disease_name,prevalence_type,prevalence_class,val_moy,geographic_area,validation_status,is_worldwide,is_validated
16564,100021,Primary plasmacytoma of the bone,Annual incidence,1-9 / 1 000 000,0.15,United States,Validated,False,True
16559,100022,Extramedullary soft tissue plasmacytoma,Annual incidence,1-9 / 1 000 000,0.10,United States,Validated,False,True
16532,100070,Progressive non-fluent aphasia,Annual incidence,1-9 / 1 000 000,0.70,Europe,Not yet validated,False,False
16482,100085,Primary hepatic neuroendocrine carcinoma,Annual incidence,1-9 / 1 000 000,0.20,Worldwide,Validated,True,True
16483,100087,Rare thyroid tumor,Annual incidence,1-9 / 100 000,3.20,Worldwide,Not yet validated,True,False
...,...,...,...,...,...,...,...,...,...
16681,99970,Dedifferentiated liposarcoma,Annual incidence,1-9 / 1 000 000,0.27,Europe,Validated,False,True
16677,99971,Well-differentiated liposarcoma,Annual incidence,1-9 / 1 000 000,0.51,Europe,Validated,False,True
16652,99976,Adenocarcinoma of the oesophagus and oesophago...,Annual incidence,1-9 / 1 000 000,0.70,Worldwide,Validated,True,True
16624,99977,Squamous cell carcinoma of the esophagus,Annual incidence,1-9 / 100 000,5.20,Worldwide,Validated,True,True


In [7]:
# load data
im_df = pd.read_csv('/wynton/group/capra/projects/dnd_project_results/data/data/master_df/cleaned_master_df_2026_08_25.csv')

In [8]:
im_df.columns

Index(['Unnamed: 0', 'hgnc_symbol', 'Gene_Group', 'GENE ID (HGNC)',
       'DISEASE LABEL', 'DISEASE ID (MONDO)', 'MOI', 'CLASSIFICATION',
       'dominant_mutation_count', 'HI Score', '%HI', 'pLI', 'LOEUF',
       'total_plp', 'missense_plp', 'nonsense_plp', 'ensg', 'chrom', 'obs_lof',
       'exp_lof', 'prior_mean', 's_het', 's_het_lower_95', 's_het_upper_95',
       'exon_disr_targetable', 'num_exon_disr_vars', 'epi_sil_targetable',
       'num_epi_sil_vars', 'excision_targetable', 'num_excision_vars',
       'num_excision_pairs', 'ss_disr_targetable', 'num_ss_disr_vars',
       'HPO_term_list', 'Approved name', 'exon_disr_spcas9_targetable',
       'epi_sil_spcas9_targetable', 'ss_disr_spcas9_targetable',
       'excision_spcas9_targetable', 'num_hets_all_strats',
       'prop_1KG_hets_all_strats', 'num_hets_exon_disr',
       'prop_1KG_hets_exon_disr', 'num_hets_epi_sil', 'prop_1KG_hets_epi_sil',
       'num_hets_ss_disr', 'prop_1KG_hets_ss_disr', 'num_hets_excision',
       'prop

In [9]:
# --- Step 5: Map im_df's MONDO IDs to OrphaCode, then join incidence ---
# Assumes im_df has a column like 'DISEASE ID (MONDO)' formatted as e.g. 'MONDO:0013212'

# strip the "MONDO:" prefix so it matches the dict's bare-digit keys
im_df['mondo_id_stripped'] = im_df['DISEASE ID (MONDO)'].str.replace('MONDO:', '', regex=False)
im_df['orpha_code'] = im_df['mondo_id_stripped'].map(mondo_to_orpha)
print(im_df['orpha_code'].isna().sum(), 'out of', len(im_df))

im_df = im_df.merge(
    incidence_best[['orpha_code', 'val_moy', 'prevalence_class', 'geographic_area']],
    on='orpha_code', how='left'
)
im_df.rename(columns={
    'val_moy': 'annual_incidence_per_10000',
    'prevalence_class': 'incidence_class',
    'geographic_area': 'incidence_geo_area'
}, inplace=True)

387 out of 660


In [13]:
# check on these numbers
temp_incidence = im_df[['hgnc_symbol','DISEASE ID (MONDO)', 'annual_incidence_per_10000','incidence_class','incidence_geo_area']].drop_duplicates()
valid_incidence = temp_incidence[~temp_incidence['annual_incidence_per_10000'].isna()]
print(valid_incidence['hgnc_symbol'].nunique())
print(len(valid_incidence['hgnc_symbol']))

28
28


In [14]:
valid_incidence.to_csv('incidence_rates.csv')

In [27]:
im_df[['hgnc_symbol','DISEASE ID (MONDO)', 'annual_incidence_per_10000','incidence_class','incidence_geo_area']].drop_duplicates()

,hgnc_symbol,DISEASE ID (MONDO),annual_incidence_per_10000,incidence_class,incidence_geo_area
0,AARS1,MONDO:0013212,NaN,NaN,NaN
1,ABCB6,MONDO:0014169,NaN,NaN,NaN
2,ABCC6,MONDO:0100091,NaN,NaN,NaN
3,ABCC8,MONDO:0015967,NaN,NaN,NaN
4,ABCC8,MONDO:0015924,0.37,1-9 / 1 000 000,Spain
...,...,...,...,...,...
655,ZIC1,MONDO:0014705,NaN,NaN,NaN
656,ZMIZ1,MONDO:0100038,NaN,NaN,NaN
657,ZMYND8,MONDO:0800439,NaN,NaN,NaN
658,ZNF292,MONDO:0100038,NaN,NaN,NaN


In [34]:
im_df[~im_df['incidence_geo_area'].isna()][['hgnc_symbol','DISEASE ID (MONDO)', 'annual_incidence_per_10000','incidence_class','incidence_geo_area']].drop_duplicates()

,hgnc_symbol,DISEASE ID (MONDO),annual_incidence_per_10000,incidence_class,incidence_geo_area
4,ABCC8,MONDO:0015924,0.370,1-9 / 1 000 000,Spain
50,ATP13A3,MONDO:0015924,0.370,1-9 / 1 000 000,Spain
89,CAV1,MONDO:0015924,0.370,1-9 / 1 000 000,Spain
96,CD46,MONDO:0016244,0.200,1-9 / 1 000 000,United States
101,CEBPA,MONDO:0018874,2.500,1-9 / 100 000,Worldwide
105,CFH,MONDO:0016244,0.200,1-9 / 1 000 000,United States
106,CFI,MONDO:0016244,0.200,1-9 / 1 000 000,United States
151,DCTN1,MONDO:0004976,1.350,1-9 / 100 000,Worldwide
232,GDF2,MONDO:0015924,0.370,1-9 / 1 000 000,Spain
233,GFAP,MONDO:0008752,0.037,<1 / 1 000 000,Japan


## Try OMIM mappings

In [30]:
omim_to_orpha = {}
for disorder in root1.iter('Disorder'):
    orpha_code_el = disorder.find('OrphaCode')
    if orpha_code_el is None:
        continue
    orpha_code = orpha_code_el.text
    for ext_ref in disorder.iter('ExternalReference'):
        source = ext_ref.find('Source')
        reference = ext_ref.find('Reference')
        if source is not None and reference is not None and source.text == 'OMIM':
            omim_to_orpha[reference.text] = orpha_code

print(f"Mapped {len(omim_to_orpha)} OMIM IDs to OrphaCodes")

Mapped 7205 OMIM IDs to OrphaCodes


In [31]:
import requests
import pandas as pd

mondo_omim_url = "https://raw.githubusercontent.com/monarch-initiative/mondo/master/src/ontology/mappings/mondo_exactmatch_omim.sssom.tsv"
r = requests.get(mondo_omim_url)
r.raise_for_status()
with open('mondo_exactmatch_omim.sssom.tsv', 'wb') as f:
    f.write(r.content)

# SSSOM files have commented header lines starting with '#' before the real header
mondo_omim_df = pd.read_csv('mondo_exactmatch_omim.sssom.tsv', sep='\t', comment='#')
print(mondo_omim_df.columns.tolist())
print(mondo_omim_df.head())

['subject_id', 'subject_label', 'predicate_id', 'object_id', 'object_label', 'mapping_justification']
      subject_id                                      subject_label  \
0  MONDO:0000070         Mycobacterium tuberculosis, susceptibility   
1  MONDO:0000208  microcephaly, short stature, and impaired gluc...   
2  MONDO:0000902  agenesis of the corpus callosum with periphera...   
3  MONDO:0000908      arrhythmogenic right ventricular dysplasia 13   
4  MONDO:0000909                            Bartter disease type 4B   

      predicate_id    object_id  \
0  skos:exactMatch  OMIM:607948   
1  skos:exactMatch  OMIM:616033   
2  skos:exactMatch  OMIM:218000   
3  skos:exactMatch  OMIM:615616   
4  skos:exactMatch  OMIM:613090   

                                        object_label  \
0      mycobacterium tuberculosis, susceptibility to   
1  microcephaly, short stature, and impaired gluc...   
2  agenesis of the corpus callosum with periphera...   
3  arrhythmogenic right ventricular 

In [32]:
# adjust column names once you've confirmed them from the printed head() above
# typical SSSOM columns: subject_id, predicate_id, object_id
mondo_to_omim = {}
for _, row in mondo_omim_df.iterrows():
    subj = row['subject_id']  # e.g. 'MONDO:0013212'
    obj = row['object_id']    # e.g. 'OMIM:601622'
    if subj.startswith('MONDO:') and obj.startswith('OMIM:'):
        mondo_id_stripped = subj.replace('MONDO:', '')
        omim_id_stripped = obj.replace('OMIM:', '')
        mondo_to_omim[mondo_id_stripped] = omim_id_stripped

print(f"Mapped {len(mondo_to_omim)} MONDO IDs to OMIM IDs")

# for genes still missing an orpha_code, try the MONDO -> OMIM -> OrphaCode path
still_missing = im_df[im_df['orpha_code'].isna()].copy()

def mondo_via_omim(mondo_id):
    omim_id = mondo_to_omim.get(mondo_id)
    if omim_id is None:
        return None
    return omim_to_orpha.get(omim_id)

im_df.loc[im_df['orpha_code'].isna(), 'orpha_code'] = (
    still_missing['mondo_id_stripped'].map(mondo_via_omim)
)

print(im_df['orpha_code'].isna().sum(), 'still unmapped out of', len(im_df))

Mapped 10037 MONDO IDs to OMIM IDs
259 still unmapped out of 660
